In [0]:
%pip install plotly xgboost lightgbm imbalanced-learn shap mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 185.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 MB 159.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.6/601.6 kB 20.7 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dis

In [0]:
# CELDA 1 — Imports
# ============================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [0]:
# CELDA 2 — Cargar CSV
# ============================================
df = pd.read_csv('/Volumes/workspace/default/fraud_data/fraud_transactions.csv')

print(f"✅ Datos cargados exitosamente")
print(f"📊 Shape: {df.shape}")
print(f"\n🎯 Distribución de clases:")
print(df['is_fraud'].value_counts())
print(f"\n📋 Columnas:")
print(list(df.columns))

✅ Datos cargados exitosamente
📊 Shape: (100000, 15)

🎯 Distribución de clases:
is_fraud
0    85000
1    15000
Name: count, dtype: int64

📋 Columnas:
['transaction_amount', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_online', 'merchant_category', 'customer_age', 'account_age_days', 'transactions_last_24h', 'avg_transaction_amt', 'amount_vs_avg_ratio', 'distance_from_home', 'failed_attempts', 'is_foreign_transaction', 'is_fraud']


In [0]:
# CELDA 3 — Verificar calidad
# ============================================
print("=" * 50)
print("DATA QUALITY CHECK")
print("=" * 50)
print(f"\nNulos por columna:")
print(df.isnull().sum())
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nPrimeras 3 filas:")
display(df.head(3))

DATA QUALITY CHECK

Nulos por columna:
transaction_amount        0
hour_of_day               0
day_of_week               0
is_weekend                0
is_online                 0
merchant_category         0
customer_age              0
account_age_days          0
transactions_last_24h     0
avg_transaction_amt       0
amount_vs_avg_ratio       0
distance_from_home        0
failed_attempts           0
is_foreign_transaction    0
is_fraud                  0
dtype: int64

Tipos de datos:
transaction_amount        float64
hour_of_day                 int64
day_of_week                 int64
is_weekend                  int64
is_online                   int64
merchant_category          object
customer_age                int64
account_age_days            int64
transactions_last_24h       int64
avg_transaction_amt       float64
amount_vs_avg_ratio       float64
distance_from_home        float64
failed_attempts             int64
is_foreign_transaction      int64
is_fraud                    int64
d

transaction_amount,hour_of_day,day_of_week,is_weekend,is_online,merchant_category,customer_age,account_age_days,transactions_last_24h,avg_transaction_amt,amount_vs_avg_ratio,distance_from_home,failed_attempts,is_foreign_transaction,is_fraud
74.46,17,1,0,1,retail,65,1646,1,84.94,0.8766,0.92,0,0,0
399.82,20,1,0,0,travel,52,3328,5,416.5,0.96,11.06,0,0,0
192.37,14,0,0,1,grocery,77,1390,4,187.17,1.0278,4.17,1,0,0


In [0]:
# CELDA 4 - Agregar columna label texto
# ============================================
df['label_text'] = df['is_fraud'].map(
    {0: 'LEGITIMATE', 1: 'FRAUD'}
)

print(f"\n✅ Columna label_text agregada")
print(df['label_text'].value_counts())


✅ Columna label_text agregada
label_text
LEGITIMATE    85000
FRAUD         15000
Name: count, dtype: int64


In [0]:
# CELDA 1 — Imports
# ============================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [0]:
# CELDA 2 - Cargar datos
# ============================================
df = pd.read_csv(
    '/Volumes/workspace/default/fraud_data/fraud_transactions.csv'
)
df['label_text'] = df['is_fraud'].map(
    {0: 'LEGITIMATE', 1: 'FRAUD'}
)
print(f"✅ {len(df):,} filas cargadas")

✅ 100,000 filas cargadas


In [0]:
# CELDA 3 - GRÁFICA 1: Distribución de clases
# ============================================
counts = df['is_fraud'].value_counts()

fig = go.Figure(go.Pie(
    labels=['Legitimate', 'Fraud'],
    values=[counts[0], counts[1]],
    hole=0.45,
    marker_colors=['#27ae60', '#e74c3c'],
    textinfo='label+percent',
    textfont_size=14
))
fig.update_layout(
    title={
        'text': '🎯 Class Distribution - 15% Fraud Rate',
        'font': {'size': 18}
    },
    height=500
)
fig.show()

print(f"\nLegitimate: {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)")
print(f"Fraud:      {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)")


Legitimate: 85,000 (85.0%)
Fraud:      15,000 (15.0%)


In [0]:
# CELDA 4 - GRÁFICA 2: Distribución de montos
# ============================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Amount Distribution by Class',
        'Amount Boxplot'
    )
)

for label, color in [('LEGITIMATE','#27ae60'),('FRAUD','#e74c3c')]:
    subset = df[df['label_text']==label]['transaction_amount']
    fig.add_trace(
        go.Histogram(
            x=subset, name=label,
            marker_color=color,
            opacity=0.7, nbinsx=50
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(
            y=subset, name=label,
            marker_color=color,
            boxpoints='outliers'
        ),
        row=1, col=2
    )

fig.update_layout(
    title='💰 Transaction Amount Analysis',
    height=500,
    barmode='overlay'
)
fig.update_xaxes(range=[0, 1000], row=1, col=1)
fig.update_yaxes(range=[0, 2000], row=1, col=2)
fig.show()

print("\n📊 Amount Statistics by Class:")
print(df.groupby('label_text')['transaction_amount'].agg(
    ['mean','median','min','max']
).round(2))


📊 Amount Statistics by Class:
              mean  median  min     max
label_text                             
FRAUD       341.21  121.92  1.0  5000.0
LEGITIMATE   86.94   54.80  5.0   500.0


In [0]:
# CELDA 5 — GRÁFICA 3: Fraude por hora
# ============================================
hourly = df.groupby(
    ['hour_of_day','label_text']
).size().reset_index(name='count')

fig = px.line(
    hourly,
    x='hour_of_day',
    y='count',
    color='label_text',
    color_discrete_map={
        'LEGITIMATE':'#27ae60',
        'FRAUD':'#e74c3c'
    },
    title='🕐 Transactions by Hour — Fraud peaks at night (0-5am)',
    labels={
        'hour_of_day':'Hour of Day',
        'count':'Transactions',
        'label_text':'Type'
    },
    markers=True
)
fig.update_layout(height=450)
fig.show()

fraud_by_hour = df[df['is_fraud']==1].groupby('hour_of_day').size()
print(f"\n🚨 Peak fraud hour: {fraud_by_hour.idxmax()}:00")
print(f"   Transactions: {fraud_by_hour.max():,}")


🚨 Peak fraud hour: 2:00
   Transactions: 1,372


In [0]:
# CELDA 6 - GRÁFICA 4: Fraude por categoría
# ============================================
category_map = {
    0:'grocery', 1:'retail',   2:'restaurant',
    3:'gas',     4:'travel',   5:'entertainment',
    6:'healthcare'
}
df['merchant_name'] = df['merchant_category'].map(category_map)

fraud_by_cat = df[df['is_fraud']==1].groupby(
    'merchant_name'
).size().reset_index(name='fraud_count').sort_values(
    'fraud_count', ascending=False
)

fig = px.bar(
    fraud_by_cat,
    x='merchant_name',
    y='fraud_count',
    title='🏪 Fraud Count by Merchant Category',
    color='fraud_count',
    color_continuous_scale='Reds',
    labels={
        'merchant_name':'Merchant Category',
        'fraud_count':'Fraud Cases'
    }
)
fig.update_layout(height=450)
fig.show()

print("\n🏪 Fraud by merchant category:")
print(fraud_by_cat.to_string(index=False))


🏪 Fraud by merchant category:
Empty DataFrame
Columns: [merchant_name, fraud_count]
Index: []


In [0]:
# CELDA 7 - GRÁFICA 5: Distancia del hogar
# ============================================
fig = go.Figure()

for label, color in [('LEGITIMATE','#27ae60'),('FRAUD','#e74c3c')]:
    subset = df[df['label_text']==label]['distance_from_home']
    fig.add_trace(go.Violin(
        y=subset,
        name=label,
        marker_color=color,
        box_visible=True,
        line_color=color,
        opacity=0.8
    ))

fig.update_layout(
    title='📍 Distance from Home - Fraud happens far from home',
    yaxis_title='Distance (km)',
    height=500,
    violinmode='overlay'
)
fig.update_yaxes(range=[0, 500])
fig.show()

print("\n📍 Average distance from home:")
print(df.groupby('label_text')['distance_from_home'].mean().round(2))


📍 Average distance from home:
label_text
FRAUD         156.58
LEGITIMATE     19.94
Name: distance_from_home, dtype: float64


In [0]:
# CELDA 8 — GRÁFICA 6: Online vs Presencial
# ============================================
online = df.groupby(
    ['is_online','label_text']
).size().reset_index(name='count')
online['channel'] = online['is_online'].map(
    {0:'In-Person', 1:'Online'}
)

fig = px.bar(
    online,
    x='channel',
    y='count',
    color='label_text',
    barmode='group',
    title='💻 Online vs In-Person — 85% of frauds are online',
    color_discrete_map={
        'LEGITIMATE':'#27ae60',
        'FRAUD':'#e74c3c'
    },
    labels={
        'channel':'Transaction Channel',
        'count':'Count',
        'label_text':'Type'
    }
)
fig.update_layout(height=450)
fig.show()

online_pct = df[df['is_fraud']==1]['is_online'].mean() * 100
print(f"\n💻 {online_pct:.1f}% of frauds are online")


💻 84.6% of frauds are online


In [0]:
# CELDA 9 - GRÁFICA 7: Intentos fallidos
# ============================================
failed = df.groupby(
    ['failed_attempts','label_text']
).size().reset_index(name='count')

fig = px.bar(
    failed,
    x='failed_attempts',
    y='count',
    color='label_text',
    barmode='group',
    title='❌ Failed Attempts — Fraudsters try multiple times',
    color_discrete_map={
        'LEGITIMATE':'#27ae60',
        'FRAUD':'#e74c3c'
    },
    labels={
        'failed_attempts':'Failed Attempts',
        'count':'Count',
        'label_text':'Type'
    }
)
fig.update_layout(height=450)
fig.show()

print("\n❌ Average failed attempts by class:")
print(df.groupby('label_text')['failed_attempts'].mean().round(2))


❌ Average failed attempts by class:
label_text
FRAUD         1.76
LEGITIMATE    0.10
Name: failed_attempts, dtype: float64


In [0]:
# CELDA 10 - GRÁFICA 8: Correlación con fraude
# ============================================
feature_cols = [
    'transaction_amount', 'hour_of_day', 'day_of_week',
    'is_weekend', 'is_online',
    'customer_age', 'account_age_days',
    'transactions_last_24h', 'avg_transaction_amt',
    'amount_vs_avg_ratio', 'distance_from_home',
    'failed_attempts', 'is_foreign_transaction'
]

correlations = df[feature_cols].corrwith(
    df['is_fraud']
).sort_values()

colors = [
    '#e74c3c' if x < -0.1
    else '#27ae60' if x > 0.1
    else '#95a5a6'
    for x in correlations
]

fig = go.Figure(go.Bar(
    x=correlations.values,
    y=correlations.index,
    orientation='h',
    marker_color=colors
))
fig.update_layout(
    title='🔍 Feature Correlation with Fraud',
    xaxis_title='Pearson Correlation',
    height=600,
    yaxis={'categoryorder':'total ascending'}
)
fig.show()

print("\n🔝 Top 5 features correlacionadas con fraude:")
print(correlations.abs().sort_values(ascending=False).head(5))


🔝 Top 5 features correlacionadas con fraude:
transactions_last_24h     0.835525
failed_attempts           0.708246
is_foreign_transaction    0.659187
distance_from_home        0.645531
transaction_amount        0.339656
dtype: float64


In [0]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                    EDA KEY FINDINGS                          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. CLASS: 15% fraud — manejable con SMOTE-Tomek            ║
║                                                              ║
║  2. AMOUNT: Fraudes tienen montos mucho más altos            ║
║     → amount_vs_avg_ratio es la feature más importante      ║
║                                                              ║
║  3. TIME: Fraude pico entre 0-5am                           ║
║     → hour_of_day tiene alto poder predictivo               ║
║                                                              ║
║  4. CHANNEL: 85% de fraudes son online                      ║
║     → is_online es muy predictivo                           ║
║                                                              ║
║  5. DISTANCE: Fraude ocurre lejos del hogar (200+ km)       ║
║     → distance_from_home es señal clara                     ║
║                                                              ║
║  6. FAILED ATTEMPTS: Fraudsters intentan varias veces       ║
║     → failed_attempts es indicador directo de fraude        ║
║                                                              ║
║  7. MERCHANT: Travel y entertainment = más fraude           ║
║     → merchant_category importa                             ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║                    EDA KEY FINDINGS                          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. CLASS: 15% fraud — manejable con SMOTE-Tomek            ║
║                                                              ║
║  2. AMOUNT: Fraudes tienen montos mucho más altos            ║
║     → amount_vs_avg_ratio es la feature más importante      ║
║                                                              ║
║  3. TIME: Fraude pico entre 0-5am                           ║
║     → hour_of_day tiene alto poder predictivo               ║
║                                                              ║
║  4. CHANNEL: 85% de fraudes son online                      ║
║     → is_online es muy predictivo                           ║
║                                                              ║
║  5. DISTANCE: Fraude ocurre 